# Discrete CPS — scikit-learn and LightGBM

CPS for count data, including CDF, PMF, PPF, coverage, and optimal inventory.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyconformal.distribution import DiscreteCrossConformalPredictiveSystem
from tinyconformal.utils import NewsvendorSolver

rng = np.random.default_rng(42)
X = rng.uniform(0, 3, size=(3000, 1))
mu = np.exp(0.7 + 0.55 * X[:, 0])
y = rng.poisson(mu)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [2]:
models = {
    "HistGradientBoosting": HistGradientBoostingRegressor(loss="poisson", max_iter=250, random_state=42),
}

scale_model = RandomForestRegressor(n_estimators=250, min_samples_leaf=8, random_state=42, n_jobs=-1)
model = HistGradientBoostingRegressor(loss="poisson", max_iter=250, random_state=42)
cps = DiscreteCrossConformalPredictiveSystem(model, scale_model, cv=5, n_jobs=-1, minimum=0).fit(X_train, y_train)
distribution = cps.predict_distribution(X_test)

display(distribution.evaluate(y_test))

,coverage,coverage_rate,interval_width_mean,mwis
0,0.50,0.620,3.240,5.980
1,0.80,0.835,6.127,8.793
2,0.90,0.917,8.150,10.550
3,0.95,0.952,9.852,12.318


## First-stage forecaster diagnostics

Before trusting the conformalized distribution, `FirstStageEvaluator` checks the
location learner on its own out-of-sample predictions (via `cross_val_predict`),
independently of the dispersion model and the conformal scaling step.

In [3]:
from sklearn.model_selection import cross_val_predict
from tinyconformal.utils import FirstStageEvaluator

oof_predictions = cross_val_predict(model, X_train, y_train, cv=5, n_jobs=-1)
first_stage_df = pd.DataFrame({"y": y_train, "y_pred": oof_predictions})

display(FirstStageEvaluator.evaluate(first_stage_df))
FirstStageEvaluator.calibration_table(first_stage_df, n_bins=10)

,Metrics
WAPE,0.3526
PBias,-0.0034
Score,0.3560
False Demand on Zero-Days (Avg Pred),2.8121
Peak Demand Deviation,-0.0183


,Calibration Bin,Count,Mean_Prediction,Mean_Observed,Mean_Residual
0,"(1.252, 2.325]",242,1.928966,2.202479,0.273513
1,"(2.325, 2.876]",238,2.648109,2.819328,0.171218
2,"(2.876, 3.464]",240,3.172215,3.237500,0.065285
3,"(3.464, 3.999]",242,3.715201,3.851240,0.136038
4,"(3.999, 4.638]",238,4.352816,4.445378,0.092562
5,"(4.638, 5.52]",241,5.076053,4.975104,-0.100949
6,"(5.52, 6.277]",242,5.922995,5.933884,0.010890
7,"(6.277, 7.462]",237,6.857761,6.831224,-0.026537
8,"(7.462, 8.685]",241,8.131656,8.431535,0.299879
9,"(8.685, 11.568]",239,9.703673,8.949791,-0.753882


## CDF, PMF, PPF, and integer quantiles

In [4]:
levels = np.array([0.1, 0.5, 0.9])
quantile_predictions = distribution.ppf(levels)

pd.DataFrame({
    "y": y_test[:10],
    "cdf_at_y": distribution.cdf(y_test[:, None])[:10],
    "pmf_at_y": distribution.pmf(y_test[:, None])[:10],
    "q10": quantile_predictions[:10, 0],
    "q50": quantile_predictions[:10, 1],
    "q90": quantile_predictions[:10, 2],
})

,y,cdf_at_y,pmf_at_y,q10,q50,q90
0,7,0.652228,0.220325,5,7,9
1,4,0.378176,0.191170,3,5,8
2,2,0.184090,0.091212,2,5,8
3,9,0.763432,0.119117,5,8,11
4,1,0.200750,0.119117,1,3,6
5,11,0.886297,0.061224,3,7,12
6,10,0.580175,0.096210,4,10,16
7,5,0.413578,0.183673,3,6,9
8,2,0.470221,0.299875,1,3,4
9,6,0.175760,0.070804,5,9,14


## Inventory solver and marginal benefit

In [5]:
decision_frame = pd.DataFrame({
    "unique_id": np.arange(len(y_test)).astype(str),
    "ds": pd.Timestamp("2026-01-01"),
    "shortage_cost": 9.0,
    "holding_cost": 1.0,
})
stock = NewsvendorSolver.optimize_distribution(
    decision_frame,
    distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
)
assert np.all(stock["y_optimal"] == np.floor(stock["y_optimal"]))
display(stock.head())
marginal_benefit = NewsvendorSolver.marginal_benefit_distribution(
    decision_frame,
    distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
    units=[0, 5, 10, 15],
)
marginal_benefit[["unique_id", "MB(k=0)", "MB(k=5)", "MB(k=10)", "MB(k=15)"]].head()

,unique_id,ds,shortage_cost,holding_cost,critical_ratio,y_optimal
0,0,2026-01-01,9.0,1.0,0.9,9.0
1,1,2026-01-01,9.0,1.0,0.9,8.0
2,2,2026-01-01,9.0,1.0,0.9,8.0
3,3,2026-01-01,9.0,1.0,0.9,11.0
4,4,2026-01-01,9.0,1.0,0.9,6.0


,unique_id,MB(k=0),MB(k=5),MB(k=10),MB(k=15)
0,0,9.0,8.112870,-0.062890,-0.987505
1,1,9.0,5.218242,-0.708455,-1.000000
2,2,9.0,4.339442,-0.408580,-0.979175
3,3,9.0,8.183673,1.365681,-0.870887
4,4,9.0,1.374011,-0.950021,-1.000000
